In [ ]:
# Load environment
using Pkg
Pkg.activate(".")
Pkg.instantiate()

In [ ]:
using Traulls, Plots, BenchmarkProfiles, DataFrames, Printf, ForwardDiff

In [ ]:
# Traulls execution

# Number of parameters for LV problems
lv_dim = [100, 500, 1000]

MAX_ITER = 500
MAX_INNER_ITER = 1000
OPT_CRIT = 1e-5
FEAS_CRIT = 1e-6


# Include problems definiton functions
# Hock Schittlowski collection
fixed_dimensions_pb = ["hs6", "hs13", "hs14", "hs16", "hs17", "hs18", "hs20", "hs22", 
"hs23", "hs26", "hs27", "hs30", "hs31", "hs32", "hs42", "hs43","hs57", "hs60", "hs61", 
"hs65", "hs70", "hs77", "hs79", "hs216", "hs227", "hs264", "hs316", "hs323", "hs337", "hs344", 
"hs345", "hs354", "hs355", "hs372", "hs373", "hs394", "hs395"]

# Luksan Vlcek
variable_dimensions_pb = ["BNST2", "BNST3", "lv501", "lv502", "lv503", "lv504", "lv511", 
"lv512", "lv513", "lv514", "lv515", "lv516", "lv517", "lv518"]

files_prefix = vcat(fixed_dimensions_pb, variable_dimensions_pb)

for prefix in files_prefix
    include("traulls_models/$(prefix).jl")
end

# First execution to compile solver
traulls(hs65())

# Dictionaries to store the results
traulls_gn = Dict{String, Traulls.TraullsResults}()
traulls_sr1_hybrid = Dict{String, Traulls.TraullsResults}()
traulls_sr1 = Dict{String, Traulls.TraullsResults}()
traulls_bfgs_hybrid = Dict{String, Traulls.TraullsResults}()
traulls_bfgs = Dict{String, Traulls.TraullsResults}()

name_instances = Vector{String}([])
# Solve problems from Hock-Schittkowski
for id in fixed_dimensions_pb
    push!(name_instances, id)
    pb = Symbol(id)
    eval(pb)()
    
    traulls_gn[id] = traulls(eval(pb)(); 
    max_iter = MAX_ITER, max_inner_iter=MAX_INNER_ITER,
    min_reltol_crit=OPT_CRIT, min_tol_feas=FEAS_CRIT)

    traulls_sr1[id] = traulls(eval(pb)(); hessian_approx=:sr1, 
    max_iter = MAX_ITER, max_inner_iter=MAX_INNER_ITER,
    min_reltol_crit=OPT_CRIT, min_tol_feas=FEAS_CRIT)

    traulls_bfgs[id] = traulls(eval(pb)(); hessian_approx = :bfgs,
    max_iter = MAX_ITER, max_inner_iter=MAX_INNER_ITER,
    min_reltol_crit=OPT_CRIT, min_tol_feas=FEAS_CRIT)
    
    traulls_sr1_hybrid[id] = traulls(eval(pb)(); hessian_approx=:hybrid_sr1, 
    max_iter = MAX_ITER, max_inner_iter=MAX_INNER_ITER,
    min_reltol_crit=OPT_CRIT, min_tol_feas=FEAS_CRIT)

    traulls_bfgs_hybrid[id] = traulls(eval(pb)(); hessian_approx=:hybrid_bfgs, 
    max_iter = MAX_ITER, max_inner_iter=MAX_INNER_ITER,
    min_reltol_crit=OPT_CRIT, min_tol_feas=FEAS_CRIT)

    @printf("\n===== %10s finished =====", id)
end

# Solve problems from Luksan-Vleck collection
for id in variable_dimensions_pb
    pb = Symbol(id)
    eval(pb)()
    
    for n in lv_dim
        id_instance = id * @sprintf("_%d", n)
        push!(name_instances, id_instance)
        
        traulls_gn[id_instance] = traulls(eval(pb)(n); hessian_approx=:gn,
        max_iter = MAX_ITER, max_inner_iter=MAX_INNER_ITER,
        min_reltol_crit=OPT_CRIT, min_tol_feas=FEAS_CRIT)

        traulls_sr1[id_instance] = traulls(eval(pb)(n); hessian_approx=:sr1,
        max_iter = MAX_ITER, max_inner_iter=MAX_INNER_ITER,
        min_reltol_crit=OPT_CRIT, min_tol_feas=FEAS_CRIT)
    
        traulls_bfgs[id_instance] = traulls(eval(pb)(n); hessian_approx = :bfgs,
        max_iter = MAX_ITER, max_inner_iter=MAX_INNER_ITER,
        min_reltol_crit=OPT_CRIT, min_tol_feas=FEAS_CRIT)

        traulls_sr1_hybrid[id_instance] = traulls(eval(pb)(n); hessian_approx=:hybrid_sr1,
        max_iter = MAX_ITER, max_inner_iter=MAX_INNER_ITER,
        min_reltol_crit=OPT_CRIT, min_tol_feas=FEAS_CRIT)
        
        traulls_bfgs_hybrid[id_instance] = traulls(eval(pb)(n); hessian_approx=:hybrid_bfgs,
        max_iter = MAX_ITER, max_inner_iter=MAX_INNER_ITER,
        min_reltol_crit=OPT_CRIT, min_tol_feas=FEAS_CRIT)

        @printf("\n===== %10s finished =====\n", id_instance)
    end
end

In [ ]:
# Form results dataframes
nb_instances = size(name_instances, 1)

res_to_df(results) = DataFrame(name = name_instances,
    n = [size(results[pb].solution, 1) for pb in name_instances],
    elapsed_time = [results[pb].elapsed_time for pb in name_instances],
    objective = [results[pb].objective for pb in name_instances],
    neval_grad = [results[pb].counters.nalgrad_eval for pb in name_instances],
    neval_residual = [results[pb].counters.nres_eval for pb in name_instances],
    neval_jac_residual = [results[pb].counters.njacres_eval for pb in name_instances],
    nouter_iter = [results[pb].counters.niter_outer for pb in name_instances],
    ninner_iter = [results[pb].counters.niter_inner for pb in name_instances],
    status = [results[pb].status for pb in name_instances])

df_gn = res_to_df(traulls_gn)
df_bfgs = res_to_df(traulls_bfgs)
df_sr1 = res_to_df(traulls_sr1)
df_hybrid_bfgs = res_to_df(traulls_bfgs_hybrid)
df_hybrid_sr1 = res_to_df(traulls_sr1_hybrid)

In [ ]:
function make_traulls_table(metric, solvers)
    T = zeros(nb_instances, size(solvers,1))
    for (i, id) in enumerate(name_instances)
        for (j, df) in enumerate(solvers)
        row_id = filter(row -> row.name == id, df)
            T[i,j] = row_id[1, :status] == Traulls.:first_order_critical ? row_id[1, metric] : Inf
        end
    end

    return T
end

In [ ]:
# Comparison of Hessian approximations
solvers_df = [df_gn, df_bfgs, df_sr1, df_hybrid_bfgs, df_hybrid_sr1]
solvers = ["Gauss-Newton", "BFGS", "SR1", "HybridBFGS", "HybridSR1"]

In [ ]:
# Show problems failures
for (i, solver) in enumerate(solvers_df)
    println("Failed instances for ", solvers[i])
    failures = filter(row -> row.status != Traulls.:first_order_critical, solver)
    println.(failures[!,:name])
end

In [ ]:


T = make_traulls_table(:elapsed_time, solvers_df)
plt_time = performance_profile(PlotsBackend(), T, solvers, 
        linestyles = [:solid, :solid, :solid, :dash, :dot], lw=2, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "elapsed time")

T = make_traulls_table(:nouter_iter, solvers_df)
plt_nouter = performance_profile(PlotsBackend(), T, solvers, 
        linestyles = [:solid, :solid, :solid, :dash, :dot], lw=2,
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of outer iterations")

T = make_traulls_table(:ninner_iter, solvers_df)
plt_ninner = performance_profile(PlotsBackend(), T, solvers, 
        linestyles = [:solid, :solid, :solid, :dash, :dot], lw=2,
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of inner iterations")

savefig(plt_time, "figures/traulls_all_hessians_time.png")
savefig(plt_nouter, "figures/traulls_all_hessians_outer.png")
savefig(plt_ninner, "figures/traulls_all_hessians_inner.png")

In [ ]:
# Comparison of Hybrid-Hessian approaches vs. Gauss-Newon
solvers_df = [df_gn, df_hybrid_bfgs, df_hybrid_sr1]
solvers = ["Gauss-Newton", "HybridBFGS", "HybridSR1"]

# Elapsed time
T = make_traulls_table(:elapsed_time, solvers_df)
plt_traulls_time = performance_profile(PlotsBackend(), T, solvers, 
        c = :black, linestyles = [:solid, :dash, :dot], lw=2, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "elapsed time")

# Number of residuals evaluation
T = make_traulls_table(:neval_residual, solvers_df)
plt_traulls_neval_res = performance_profile(PlotsBackend(), T, solvers, 
        c = :black, linestyles = [:solid, :dash, :dot], lw=2, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of residuals evaluations")

# Number of gradient evaluation
T = make_traulls_table(:neval_grad, solvers_df)
plt_traulls_neval_grad = performance_profile(PlotsBackend(), T, solvers, 
       c = :black, linestyles = [:solid, :dash, :dot], lw=2, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of gradient evaluations")

In [ ]:
display(plt_traulls_time)
savefig(plt_traulls_time, "figures/gn_vs_hybrid_time_profile.png")

In [ ]:
display(plt_traulls_neval_res)
savefig(plt_traulls_neval_res, "figures/gn_vs_hybrid_reseval_profile.png")

In [ ]:
display(plt_traulls_neval_grad)
savefig(plt_traulls_neval_grad, "figures/gn_vs_hybrid_gradeval_profile.png")

# Comparison against other solvers

In [ ]:
# Running Percival.jl and Ipopt.jl
using NLPModelsIpopt, Percival, SolverBenchmark, NLSProblems

problems = [NLSProblems.hs06(), NLSProblems.hs13(), NLSProblems.hs14(), NLSProblems.hs16(), NLSProblems.hs17(), NLSProblems.hs18(),
    NLSProblems.hs20(), NLSProblems.hs22(), NLSProblems.hs23(), NLSProblems.hs26(), NLSProblems.hs27(), NLSProblems.hs30(), 
    NLSProblems.hs31(), NLSProblems.hs32(), NLSProblems.hs42(), NLSProblems.hs43(), NLSProblems.hs57(), NLSProblems.hs60(), 
    NLSProblems.hs61(), NLSProblems.hs65(), NLSProblems.hs70(), NLSProblems.hs77(), NLSProblems.hs79(), 
    tp216(), tp227(), tp264(), tp316(), tp323(), tp337(), tp344(),
    tp345(), tp354(), tp355(), tp372(), tp373(), tp394(), tp395(),
    NLSProblems.BNST2(100), NLSProblems.BNST2(500), NLSProblems.BNST2(1000),
    NLSProblems.BNST3(100), NLSProblems.BNST3(500), NLSProblems.BNST3(1000),
    LVcon501(100), LVcon501(500), LVcon501(1000),
    LVcon502(100), LVcon502(500), LVcon502(1000), 
    LVcon503(100), LVcon503(500), LVcon503(1000),
    LVcon504(100), LVcon504(500), LVcon504(1000),
    LVcon511(100), LVcon511(500), LVcon511(1000),
    LVcon512(100), LVcon512(500), LVcon512(1000),
    LVcon513(100), LVcon513(500), LVcon513(1000),
    LVcon514(100), LVcon514(500), LVcon514(1000),
    LVcon515(100), LVcon515(500), LVcon515(1000),
    LVcon516(100), LVcon516(500), LVcon516(1000),
    LVcon517(100), LVcon517(500), LVcon517(1000),
    LVcon518(100), LVcon518(500), LVcon518(1000)]

small_batch = [NLSProblems.hs06(), NLSProblems.hs13(), NLSProblems.hs14(), NLSProblems.hs16(), NLSProblems.hs17(), NLSProblems.hs18(),
    NLSProblems.hs20(), NLSProblems.hs22(),
            NLSProblems.hs23(), NLSProblems.hs26(), NLSProblems.hs27(), NLSProblems.hs30(), NLSProblems.hs31(), 
    NLSProblems.hs32(), NLSProblems.hs42(), NLSProblems.hs43(), NLSProblems.hs57(), NLSProblems.hs60(), NLSProblems.hs61(),
    NLSProblems.hs65(), NLSProblems.hs70(), NLSProblems.hs77(), NLSProblems.hs79(), 
    tp216(), tp227(), tp264(), tp316(), tp323(), tp337(), tp344(),
    tp345(), tp354(), tp355(), tp372(), tp373(), tp394(), tp395(),
    NLSProblems.BNST2(100), NLSProblems.BNST3(100),
    LVcon501(100),
    LVcon502(100),
    LVcon503(100),
    LVcon504(100),
    LVcon511(100),
    LVcon512(100),
    LVcon513(100),
    LVcon514(100),
    LVcon515(100),
    LVcon516(100),
    LVcon517(100),
    LVcon518(100)]

solvers = Dict(:percival => model -> percival(model; inity = true,
                                              atol=1e-5, rtol = 1e-5, ctol=1e-6, ω_min = 1e-5,
                                              max_time = 30.0, subsolver_max_iter=1000),
               :ipopt => model -> ipopt(model; tol=1e-5, max_iter = 1000, nlp_scaling_method="none",
        dual_inf_tol = Inf,
        constr_viol_tol = Inf,
        compl_inf_tol = Inf,
        acceptable_iter = 0,
        print_level=0))

stats = bmark_solvers(solvers, problems)

In [ ]:
for solver in keys(solvers)
    stats[solver][!, :name] .= name_instances
end
stats[:percival]

In [ ]:
# Insert numbe rof gradient evaluations for Ipopt
# Equals the number of jacobian evaluation minus 1
stats[:ipopt][!, :neval_grad] .= stats[:ipopt][!, :neval_jac_residual] .- 1
show(stats[:ipopt][!,[:name, :status]], allrows=true)

In [ ]:
solvers = ["Traulls.jl", "Ipopt.jl", "Percival.jl"]

function make_performance_table(stats_traulls, stats_ipopt, stats_percival, name_instances, metric)

    nb_instances = size(name_instances, 1)
    T = zeros(nb_instances, 3)

    for (i, pb) in enumerate(name_instances)
        row_traulls = filter(row -> row.name == pb, stats_traulls)
        T[i, 1] = row_traulls[1, :status] == Traulls.:first_order_critical ? row_traulls[1, metric] : Inf
        
        row_ipopt = filter(row -> row.name == pb, stats_ipopt)
        T[i, 2] = row_ipopt[1, :internal_msg] == :Solve_Succeeded ? row_ipopt[1, metric] : Inf
        
        row_percival = filter(row -> row.name == pb, stats_percival)
        T[i, 3] = row_percival[1, :status] == :first_order ? row_percival[1, metric] : Inf
    end

    return T
end

performance_table(metric) = make_performance_table(df_hybrid_sr1, stats[:ipopt], stats[:percival], name_instances, metric)

In [ ]:
# Elapsed time
T = performance_table(:elapsed_time)
plt_elapsed_time = performance_profile(PlotsBackend(), T, solvers, 
        c = :black, linestyles = [:dot, :solid, :dash], lw=2, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "elapsed time")

# Residual evaluation
T = performance_table(:neval_residual)
plt_neval_res = performance_profile(PlotsBackend(), T, solvers, 
        c = :black, linestyles = [:dot, :solid, :dash], lw=2, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of residuals evaluations")

# Gradient evaluations
T = performance_table(:neval_grad)
plt_neval_grad = performance_profile(PlotsBackend(), T, solvers, 
        c = :black, linestyles = [:dot, :solid, :dash], lw=2, 
    legend=:bottomright, xaxis = "τ (logscale)", yaxis = "ρ(τ)", title = "number of gradient evaluations");

In [ ]:
display(plt_elapsed_time)
savefig(plt_elapsed_time, "figures/traulls_vs_others_time_profile.png")

In [ ]:
display(plt_neval_res)
savefig(plt_neval_res, "figures/traulls_vs_others_reseval_profile.png")

In [ ]:
display(plt_neval_grad)
savefig(plt_neval_grad, "figures/traulls_vs_others_gradeval_profile.png")